In [2]:
# Cell 1 — Remove unused df load, or keep for inspection only
import pandas as pd

df = pd.read_csv("dataset_facebook-comments-scraper_2026-02-20_23-21-35-548.csv")
print(df.shape)
df.head()  # keep only if you want to inspect raw data

(63, 5)


,postTitle,postDescription,text,likesCount,facebookUrl
0,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Ot jog yy te oy lg mix jg2 ey ng,0,https://web.facebook.com/share/p/1Fz4HU3JRM/
1,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,"S25 ultra still cannot use 5G, CS service stil...",0,https://web.facebook.com/share/p/1Fz4HU3JRM/
2,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Smart ឡូវ សុីលុយមេះបងបញ្ចូល6$ ប្រើបានតែ12ថ្ងៃអ...,0,https://web.facebook.com/share/p/1Fz4HU3JRM/
3,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Smart Axiata សុំដាក់ 5G នៅ ស្រុកឱរ៉ាល់ ខេត្តកំ...,1,https://web.facebook.com/share/p/1Fz4HU3JRM/
4,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,បងនៅ ភូមិចំការស្លែង ស្រុកអង្គស្នួល ខេត្តកណ្តាល...,0,https://web.facebook.com/share/p/1Fz4HU3JRM/


In [3]:
# Install khmernltk for Khmer word segmentation (run once)
# %pip install khmernltk


In [4]:
import pandas as pd
import re
import emoji
from khmernltk import word_tokenize as khmer_word_tokenize
from sentence_transformers import SentenceTransformer

class FacebookCommentPreprocessor:
    """
    Preprocess Facebook comments for transformer-based embeddings and clustering.
    Supports both Khmer and English text.
    (Improved approach only – no TF-IDF, no stopword removal)
    """

    def __init__(self, data):
        """
        Args:
            data (pd.DataFrame or str): DataFrame or path to CSV file
        """
        if isinstance(data, str):
            self.dataset = pd.read_csv(data)
        else:
            self.dataset = data.copy()

        self.cleaning_dataset = None

        # Emoji sentiment mapping
        self.EMOJI_SENTIMENT_MAP = {
            "crying_face": "EMO_SAD",
            "loudly_crying_face": "EMO_SAD",
            "broken_heart": "EMO_SAD",
            "red_heart": "EMO_LOVE",
            "heart": "EMO_LOVE",
            "smiling_face_with_heart_eyes": "EMO_LOVE",
            "grinning_face": "EMO_HAPPY",
            "smiling_face": "EMO_HAPPY",
            "face_with_tears_of_joy": "EMO_HAPPY",
            "angry_face": "EMO_ANGRY",
            "pouting_face": "EMO_ANGRY",
        }

        self._normalize_columns()

    def _normalize_columns(self):
        """Normalize column names and rename 'text' -> 'comment_text' if needed."""
        self.dataset.columns = (
            self.dataset.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_", regex=False)
        )
        # Rename scraped 'text' column to standard 'comment_text'
        if 'text' in self.dataset.columns and 'comment_text' not in self.dataset.columns:
            self.dataset = self.dataset.rename(columns={'text': 'comment_text'})

    # ------------------------------------------------------------------
    # Khmer language helpers
    # ------------------------------------------------------------------

    def is_khmer_text(self, text):
        """Return True if the text is predominantly Khmer (Unicode U+1780–U+17FF)."""
        if not isinstance(text, str) or len(text) == 0:
            return False
        khmer_chars = sum(1 for c in text if '\u1780' <= c <= '\u17FF')
        return (khmer_chars / len(text)) > 0.3

    def khmer_segment(self, text):
        """
        Segment Khmer text into space-separated tokens using khmernltk.
        English text is returned unchanged.
        """
        if self.is_khmer_text(text):
            tokens = khmer_word_tokenize(text)
            return ' '.join(t for t in tokens if t.strip())
        return text

    # ------------------------------------------------------------------
    # Filtering
    # ------------------------------------------------------------------

    def has_min_words(self, text, min_words=3):
        """
        Check if text has at least min_words words.
        Uses khmernltk tokenisation for Khmer text.
        """
        if pd.isna(text) or text.strip() == "":
            return False
        if self.is_khmer_text(text):
            tokens = khmer_word_tokenize(text)
            return len([t for t in tokens if t.strip()]) >= min_words
        return len(text.split()) >= min_words

    # Fix filter_comments to also keep original comment_text for Gemini
    def filter_comments(self, min_words=3):
        """Remove short or empty comments while keeping metadata."""
        # Keep ALL columns, not just comment_text
        self.cleaning_dataset = self.dataset.copy()

        self.cleaning_dataset = self.cleaning_dataset[
            self.cleaning_dataset['comment_text'].apply(
                lambda x: self.has_min_words(x, min_words)
            )
        ].reset_index(drop=True)

        return self

    # ------------------------------------------------------------------
    # Text cleaning
    # ------------------------------------------------------------------

    def remove_urls(self, text):
        """Remove URLs."""
        return re.sub(r'https?://\S+|www\.\S+', '', text)

    def remove_timestamps(self, text):
        """Remove timestamps like 01:23 or 01:23:45."""
        return re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', '', text)

    def emoji_to_sentiment(self, text):
        """Convert emojis to sentiment tokens."""
        text = emoji.demojize(text, delimiters=(" ", " "))
        for emoji_name, token in self.EMOJI_SENTIMENT_MAP.items():
            text = re.sub(rf"\b{emoji_name}\b", token, text)
        return text

    def clean_text(self):
        """
        Clean comments (transformer-friendly).
        Applies Khmer word segmentation before lowercasing so tokens are preserved.
        """
        self.cleaning_dataset['comment_clean'] = (
            self.cleaning_dataset['comment_text']
            .apply(self.remove_urls)
            .apply(self.emoji_to_sentiment)
            .apply(self.remove_timestamps)
            .apply(self.khmer_segment)   # segment Khmer words with spaces
            .str.lower()
            .str.strip()
        )

        return self

    # ------------------------------------------------------------------
    # Pipeline
    # ------------------------------------------------------------------

    def preprocess_all(self, min_words=3):
        """Full preprocessing pipeline."""
        self.filter_comments(min_words)
        self.clean_text()
        return self

    def generate_embeddings(self, model_name='paraphrase-multilingual-MiniLM-L12-v2', show_progress=True):
        """
        Generate sentence embeddings using a multilingual model
        that supports both Khmer and English text.
        """
        model = SentenceTransformer(model_name)

        embeddings = model.encode(
            self.cleaning_dataset['comment_clean'].tolist(),
            show_progress_bar=show_progress,
            convert_to_numpy=True
        )

        print(f"Embeddings shape: {embeddings.shape}")
        return embeddings

    # ------------------------------------------------------------------
    # Accessors
    # ------------------------------------------------------------------

    def get_cleaned_dataset(self):
        """Return cleaned dataset."""
        return self.cleaning_dataset

    def get_original_dataset(self):
        """Return original dataset."""
        return self.dataset


d:\2024\anaconda\envs\cuda_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def preprocess_facebook_comments(
    data,
    min_words=3,
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    show_progress=True
):
    """
    Fully preprocess Facebook comments using FacebookCommentPreprocessor.
    Supports both Khmer and English text via khmernltk word segmentation
    and a multilingual sentence embedding model.

    Args:
        data (pd.DataFrame or str): DataFrame or path to CSV file
        min_words (int): Minimum number of words per comment
        model_name (str): SentenceTransformer model name
        show_progress (bool): Show embedding progress bar

    Returns:
        cleaned_df (pd.DataFrame): Cleaned comments
        embeddings (np.ndarray): Sentence embeddings
    """

    preprocessor = FacebookCommentPreprocessor(data)

    preprocessor.preprocess_all(min_words=min_words)

    cleaned_df = preprocessor.get_cleaned_dataset()

    embeddings = preprocessor.generate_embeddings(
        model_name=model_name,
        show_progress=show_progress
    )

    return cleaned_df, embeddings


In [6]:
# Apply preprocessing using CSV file path
# Uses multilingual model to handle both Khmer and English comments
cleaned_df, embeddings = preprocess_facebook_comments(
    data="dataset_facebook-comments-scraper_2026-02-20_23-21-35-548.csv",
    min_words=3,
    model_name="paraphrase-multilingual-MiniLM-L12-v2",
    show_progress=True
)

print(f"Total comments after filtering: {len(cleaned_df)}")
cleaned_df.head()


| 2026-02-24 21:09:03,215 | INFO | khmer-nltk | Loaded model from d:\2024\anaconda\envs\cuda_env\lib\site-packages\khmernltk\word_tokenize\sklearn_crf_ner_10000.sav |
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 315.15it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:01<00:00,  1.94it/s]

Embeddings shape: (38, 384)
Total comments after filtering: 38


,posttitle,postdescription,comment_text,likescount,facebookurl,comment_clean
0,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Ot jog yy te oy lg mix jg2 ey ng,0,https://web.facebook.com/share/p/1Fz4HU3JRM/,ot jog yy te oy lg mix jg2 ey ng
1,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,"S25 ultra still cannot use 5G, CS service stil...",0,https://web.facebook.com/share/p/1Fz4HU3JRM/,"s25 ultra still cannot use 5g, cs service stil..."
2,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Smart ឡូវ សុីលុយមេះបងបញ្ចូល6$ ប្រើបានតែ12ថ្ងៃអ...,0,https://web.facebook.com/share/p/1Fz4HU3JRM/,smart ឡូវ សុី លុយ មេះ បង បញ្ចូល 6$ ប្រើ បាន តែ...
3,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,Smart Axiata សុំដាក់ 5G នៅ ស្រុកឱរ៉ាល់ ខេត្តកំ...,1,https://web.facebook.com/share/p/1Fz4HU3JRM/,smart axiata សុំ ដាក់ 5g នៅ ស្រុក ឱរ៉ាល់ ខេត្ត...
4,សមិទ្ធផលរបស់ Smart ក្នុងខែមករា 2026 🌐🎊\nត្រឹមត...,NaN,បងនៅ ភូមិចំការស្លែង ស្រុកអង្គស្នួល ខេត្តកណ្តាល...,0,https://web.facebook.com/share/p/1Fz4HU3JRM/,បង នៅ ភូមិ ចំការស្លែង ស្រុក អង្គស្នួល ខេត្ត កណ...


In [7]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# 1. Determine Dynamic K using Silhouette Score
max_k = min(10, len(embeddings) - 1) # Ensure k doesn't exceed dataset size
best_k = 2
best_score = -1
best_kmeans = None

print("🔍 Evaluating optimal cluster counts...")
for k in range(2, max_k + 1):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    
    if score > best_score:
        best_score = score
        best_k = k
        best_kmeans = kmeans

print(f"✅ Optimal number of clusters chosen: {best_k} (Silhouette Score: {best_score:.4f})")

# 2. Fit and predict clusters
cleaned_df['cluster'] = best_kmeans.labels_

# Display cluster distribution
print("\nCluster distribution:")
print(cleaned_df['cluster'].value_counts().sort_index())

🔍 Evaluating optimal cluster counts...
✅ Optimal number of clusters chosen: 2 (Silhouette Score: 0.2080)

Cluster distribution:
cluster
0    29
1     9
Name: count, dtype: int64


In [8]:
# import os
# import math
# import google.generativeai as genai
# from sklearn.metrics.pairwise import cosine_similarity
# from dotenv import load_dotenv

# load_dotenv()
# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
# gemini_model = genai.GenerativeModel("gemini-2.5-flash")

# def get_representative_comments(df, embeddings, kmeans_model, cluster_id, min_samples=5, max_samples=30):
#     """Extracts top N comments closest to the cluster centroid."""
#     cluster_indices = df.index[df['cluster'] == cluster_id].tolist()
#     cluster_size = len(cluster_indices)

#     # Dynamic proportional sizing (5% of cluster, bounded by min/max)
#     n_samples = max(min_samples, min(max_samples, math.ceil(0.05 * cluster_size)))
#     n_samples = min(n_samples, cluster_size) # Prevent over-sampling

#     # Get embeddings for this cluster and its centroid
#     cluster_embeddings = embeddings[cluster_indices]
#     centroid = kmeans_model.cluster_centers_[cluster_id].reshape(1, -1)

#     # Calculate cosine similarity
#     similarities = cosine_similarity(cluster_embeddings, centroid).flatten()

#     # Get original indices of the most similar comments
#     top_indices_relative = similarities.argsort()[::-1][:n_samples]
#     top_indices_absolute = [cluster_indices[i] for i in top_indices_relative]

#     return df.loc[top_indices_absolute]

# def summarize_comment_cluster(representative_df, cluster_id):
#     if representative_df.empty:
#         return f"Cluster {cluster_id}: No comments available."

#     # Format text to include post origin context
#     formatted_comments = []
#     for _, row in representative_df.iterrows():
#         # Fallback to "Unknown Post" if posttitle is missing
#         post_context = str(row.get('posttitle', 'Unknown Post')).strip()[:60] 
#         text = row['comment_text']
#         formatted_comments.append(f"[From Post: {post_context}...] {text}")

#     comments_text = "\n- ".join(formatted_comments)

#     prompt = f"""
# You are a social media insight analyst helping a business understand customer feedback across multiple Facebook posts.

# CLUSTER ID: {cluster_id}

# TASK:
# The following comments belong to ONE thematic opinion cluster. Note that they may originate from different posts.
# Analyze the shared perspective and provide insights. If an opinion is heavily tied to a specific post, mention it.

# OUTPUT FORMAT:
# Topic: <short label>
# Customer Perspective: <2-3 sentences>
# Overall Sentiment: <Positive / Negative / Mixed>
# Business Insight: <What this means>
# Suggested Action: <1 practical action>
# Representative Keywords: <3-5 keywords>

# COMMENTS:
# {comments_text}
# """
#     try:
#         response = gemini_model.generate_content(prompt)
#         return response.text
#     except Exception as e:
#         return f"❌ Error in cluster {cluster_id}: {e}"

# # --- Execution ---
# print("✅ Models loaded. Starting summarization...\n")

# for cluster_id in sorted(cleaned_df['cluster'].unique()):
#     # 1. Get the best comments based on centroid similarity
#     rep_df = get_representative_comments(cleaned_df, embeddings, best_kmeans, cluster_id)
    
#     print(f"{'='*60}\nCLUSTER {cluster_id} ANALYSIS (Analyzed {len(rep_df)} representative comments)\n{'='*60}")
    
#     # 2. Summarize
#     summary = summarize_comment_cluster(rep_df, cluster_id)
#     print(summary)
#     print("\n")

In [9]:
import joblib
from sentence_transformers import SentenceTransformer

# 1. Save the K-Means Model
# Assuming 'best_kmeans' is the model you generated in the previous steps
joblib.dump(best_kmeans, 'kmeans_clustering_model.pkl')
print("✅ K-Means model saved to 'kmeans_clustering_model.pkl'")

# 2. Save the Sentence Transformer locally 
# This prevents your API from downloading the model from HuggingFace every time it boots up.
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
model.save('./models/sentence_transformer_local')
print("✅ Transformer model saved to './models/sentence_transformer_local'")

✅ K-Means model saved to 'kmeans_clustering_model.pkl'


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 375.51it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]

✅ Transformer model saved to './models/sentence_transformer_local'
